# Preprocessing pipelines

Transforms are the preprocessing layer: composable, non-mutating operations on a single scene. Each one operates on a **single-scene** `dict` (one sample, before collation), is single-purpose, and never mutates its input. You chain them with `Compose`, and every non-trivial transform has a tensor-level twin under `torch_pointcloud.transforms.functional`.

This notebook builds intuition step by step on a synthetic scene, so every cell runs on CPU. The [Transforms gallery](../transforms/overview.md) shows before/after pictures of the full catalog.

In [ ]:
import torch

import torch_pointcloud.transforms as T

torch.manual_seed(0)

In [ ]:
import matplotlib.pyplot as plt


def show_cloud(pos, color=None, *, ax=None, title=None, size=6, cmap="viridis"):
    """Scatter a point cloud. `pos` is (N, 3); `color` is per-point RGB, a label vector, or None."""
    if ax is None:
        ax = plt.figure(figsize=(4, 4)).add_subplot(projection="3d")

    p = pos.detach().cpu().numpy()
    c = color.detach().cpu().numpy() if torch.is_tensor(color) else color
    kw = {} if c is None else {"cmap": cmap}
    ax.scatter(p[:, 0], p[:, 1], p[:, 2], c=c, s=size, depthshade=False, linewidths=0, **kw)
    ax.set_box_aspect((1, 1, 1))
    ax.set_axis_off()
    if title:
        ax.set_title(title, fontsize=10)
    return ax

## A scene is a dict

We use the standard keys: `pos` for $(N, 3)$ coordinates and `color` for $(N, 3)$ RGB. Here is a synthetic "wall + floor" scene with a colour gradient, so transforms are easy to see.

In [ ]:
n = 8000
floor = torch.rand(n // 2, 3) * torch.tensor([4.0, 4.0, 0.05])
wall = torch.rand(n // 2, 3) * torch.tensor([4.0, 0.05, 2.5])
pos = torch.cat([floor, wall])
color = (pos - pos.min(0).values) / (pos.max(0).values - pos.min(0).values)  # position -> RGB

scene = {"pos": pos, "color": color}
{k: tuple(v.shape) for k, v in scene.items()}

In [ ]:
show_cloud(scene["pos"], color=scene["color"], title="input scene");

## One transform at a time

A transform is constructed with the `keys` it acts on, then called on the dict. It returns a **new** dict; keys it does not touch pass through untouched. `Rescale(method="centroid")` centers the cloud and scales it into the unit sphere:

In [ ]:
rescaled = T.Rescale(keys="pos", method="centroid")(scene)

print("input  center / radius:", scene["pos"].mean(0).round(decimals=2).tolist(), "/", round(scene["pos"].norm(dim=1).max().item(), 2))
print("output center / radius:", rescaled["pos"].mean(0).round(decimals=2).tolist(), "/", round(rescaled["pos"].norm(dim=1).max().item(), 2))
print("color untouched:", torch.equal(rescaled["color"], scene["color"]))

When several keys are passed together, they stay in correspondence. `RandomSample` draws the same indices for every listed key, so `pos` and `color` shrink to the same 2048 rows:

In [ ]:
sampled = T.RandomSample(keys=("pos", "color"), num_samples=2048)(rescaled)
{k: tuple(v.shape) for k, v in sampled.items()}

## Augmentations

Augmentations are random and take a probability `p`. With `p=1.0` they always fire (handy for a demo). Below: a 30 degrees rotation about the vertical axis and a per-point jitter.

In [ ]:
rotated = T.RandomRotate(keys="pos", angle_range=(30.0, 30.0), axis=2, p=1.0)(sampled)
jittered = T.RandomJitter(keys="pos", sigma=0.02, clip=0.05, p=1.0)(rotated)

fig = plt.figure(figsize=(9, 3))
show_cloud(sampled["pos"], color=sampled["color"], ax=fig.add_subplot(131, projection="3d"), title="sampled")
show_cloud(rotated["pos"], color=rotated["color"], ax=fig.add_subplot(132, projection="3d"), title="+ rotate")
show_cloud(jittered["pos"], color=jittered["color"], ax=fig.add_subplot(133, projection="3d"), title="+ jitter");

Colour transforms act on the `color` key. They expect $[0, 1]$ floats by default (pass `int_color=True` for $[0, 255]$).

In [ ]:
jit = T.RandomColorJitter(keys="color", brightness=0.5, contrast=0.5, saturation=0.5, p=1.0)(scene)
show_cloud(scene["pos"], color=jit["color"], title="RandomColorJitter");

## Compose a pipeline

`Compose` chains transforms into one callable. A typical training pipeline normalizes geometry, subsamples to a fixed budget, then augments:

In [ ]:
train_pipeline = T.Compose([
    T.Rescale(keys="pos", method="centroid"),
    T.RandomSample(keys=("pos", "color"), num_samples=2048),
    T.RandomFlip(keys="pos", axes=[0, 1], p=0.5),
    T.RandomScale(keys="pos", scale_range=(0.9, 1.1)),
    T.RandomJitter(keys="pos", sigma=0.01, clip=0.05),
])

out = train_pipeline({"pos": pos.clone(), "color": color.clone()})
{k: tuple(v.shape) for k, v in out.items()}

At evaluation you keep the deterministic steps and drop the random ones, so results are reproducible:

In [ ]:
eval_pipeline = T.Compose([
    T.Rescale(keys="pos", method="centroid"),
    T.RandomSample(keys=("pos", "color"), num_samples=2048),
])
tuple(eval_pipeline({"pos": pos.clone(), "color": color.clone()})["pos"].shape)

## Reproducibility

Random transforms accept a `torch.Generator`, so a pipeline can be made deterministic without touching the global RNG. Two draws from the same seed match:

In [ ]:
g1 = torch.Generator().manual_seed(42)
g2 = torch.Generator().manual_seed(42)
a = T.RandomSample(keys="pos", num_samples=512, generator=g1)({"pos": pos.clone()})["pos"]
b = T.RandomSample(keys="pos", num_samples=512, generator=g2)({"pos": pos.clone()})["pos"]
print("identical draws:", torch.equal(a, b))

## The functional layer

If you already hold a tensor and do not want a dict, call the functions in `torch_pointcloud.transforms.functional` directly. They are the same operations the class transforms wrap.

In [ ]:
import torch_pointcloud.transforms.functional as F

p = torch.randn(1000, 3)
p = F.shift(p, method="bbox", axes=[0, 1])  # center X and Y on the bbox midpoint
p = F.shift(p, method="min", axes=[2])       # drop Z so the floor sits at 0
mask = F.sphere_mask(p, center=[0.0, 0.0, 0.0], radius=2.0)
print("shifted:", tuple(p.shape), "| kept by sphere mask:", int(mask.sum()))

## Next steps

- Browse every transform with before/after pictures in the [Transforms gallery](../transforms/overview.md).
- Each pretrained checkpoint records its own pipeline: `create_model(..., return_info=True)["transforms"]`.
- Plug a pipeline into a dataset in [Use your own data](04-custom-dataset.md).